In [2]:
require("data.table")

# 1. Buscar todos los archivos prediccion.txt dentro de la carpeta Final y subcarpetas
archivos <- list.files(
  path = "/home/ds/buckets/b1/exp/Final", 
  pattern = "^prediccion\\.txt$", 
  full.names = TRUE, 
  recursive = TRUE
)

# Filtrar estrictamente para incluir solo los métodos elegidos (dolar_oficial y deflacion)
archivos_validos <- archivos[grepl("dolar_oficial|deflacion", archivos)]

cat("Total de modelos incluidos en el ensemble:", length(archivos_validos), "\n")

if (length(archivos_validos) == 0) {
  stop("¡Atención! No se encontraron archivos de predicción válidos en las rutas especificadas.")
}

# 2. Cargar, acumular y promediar las probabilidades de todos los archivos
dt_ensemble <- NULL

for (arch in archivos_validos) {
  cat("Agregando predicciones de:", arch, "\n")
  dt_temp <- fread(arch)
  
  if (is.null(dt_ensemble)) {
    dt_ensemble <- dt_temp[, list(numero_de_cliente, prob_acum = prob)]
  } else {
    dt_ensemble[, prob_acum := prob_acum + dt_temp$prob]
  }
}

# Calcular el promedio aritmético de las probabilidades
dt_ensemble[, prob_promedio := prob_acum / length(archivos_validos)]

# 3. Ordenar de mayor a menor probabilidad y aplicar el corte fijo de 1800 envíos
setorder(dt_ensemble, -prob_promedio)

dt_ensemble[, Predicted := 0L]
#dt_ensemble[1:1800, Predicted := 1L]
dt_ensemble[prob_promedio >= 0.025, Predicted := 1L]

cat("Cantidad de estímulos asignados (1):", sum(dt_ensemble$Predicted == 1), "\n")


# 4. Guardar el archivo definitivo para la subida a Kaggle
archivo_final <- "/home/ds/buckets/b1/exp/Final/SUBMIT_FINAL_ENSEMBLE_1800.csv"

fwrite(
  dt_ensemble[, list(numero_de_cliente, Predicted)], 
  file = archivo_final, 
  sep = ","
)

cat("\n¡Éxito! Ensemble Final generado y guardado en:\n", archivo_final, "\n")

Total de modelos incluidos en el ensemble: 13 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF100333deflacion/prediccion.txt 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF100333dolar_oficial/prediccion.txt 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF100519deflacion/prediccion.txt 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF100519dolar_oficial/prediccion.txt 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF100523dolar_oficial/prediccion.txt 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF300331deflacion/prediccion.txt 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF300331dolar_oficial/prediccion.txt 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF500333deflacion/prediccion.txt 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF500333dolar_oficial/prediccion.txt 
Agregando predicciones de: /home/ds/buckets/b1/exp/Final/WF700331deflacion/prediccion.txt 
Agregando predicciones 